# Gán nhãn một đợt bằng cấu hình đã khóa (NLP_N12)

Mỗi lần chạy notebook này gán **một đợt** (khoảng 9.500–10.300 bình luận, khoảng 6–6,5 giờ) trên **Kaggle, GPU T4 x2**.

**Cấu hình đã khóa ngày 27/11/2026. Không sửa notebook hay code trong gói.**
- Mô hình `Qwen/Qwen3-14B-AWQ`, vLLM 0.34.0.
- Prompt `ban5-en-tach-p3`, sha `e4cc4216433b`; ô 2 kiểm tra mã này.
- Tắt suy nghĩ, nhiệt độ 0, 16 luồng.

Bảng phân công nằm trong `data/llm/dot/phan_cong.csv` của gói:

| Người | Đợt |
|---|---|
| Hảo | dot_01, dot_05, dot_09, dot_13 (test) |
| Sinh | dot_02, dot_06, dot_10, dot_14 (test) |
| Triệu | dot_03, dot_07, dot_11 (val) |
| Long | dot_04, dot_08, dot_12 (val) |

Mỗi đợt khoảng 6–6,5 giờ GPU; 4 đợt khoảng 25 giờ, vẫn dưới 30 giờ mỗi tuần của Kaggle.

**Lần đầu (mỗi người làm một lần)**
1. Tài khoản Kaggle phải xác minh số điện thoại, để bật được GPU và Internet.
2. Nhận `goi_<tên>.zip` từ Triệu qua kênh riêng của nhóm. Không gửi qua kênh công khai.
3. Kaggle › Datasets › New Dataset › tải `goi_<tên>.zip` lên, **để Private**. Kaggle có thể tự giải nén gói; notebook vẫn chạy được.
4. Import notebook này (File › Import Notebook). Settings: **GPU T4 x2**, **Internet On**. Add Input là dataset vừa tạo.

**Mỗi đợt**
1. Ở ô 1, đổi `DOT` thành đợt cần chạy, ví dụ `"dot_05"`.
2. Bấm **Save Version › Save & Run All (Commit)**. Notebook chạy nền: tắt máy hay đóng trình duyệt vẫn chạy, và file Output được giữ lại.
3. Khi xong, vào tab Output của phiên bản đó, tải `ket_qua_<đợt>.zip` về, gửi cho Triệu qua kênh riêng của nhóm.
4. Mỗi lúc chỉ chạy một đợt; xong đợt này mới chạy đợt sau.

**Nếu đợt chưa xong.** Ô 7 báo "CHƯA XONG" khi hết giờ (sau `DUNG_SAU_PHUT` phút) hoặc khi còn câu lỗi. Khi đó:
1. Mở lại notebook, Add Input › chọn Output của phiên bản vừa chạy (Notebook Output Files).
2. Giữ nguyên `DOT`, chạy lại.

Ô 3 tìm file kết quả dở trong Input và chỉ gán các câu còn thiếu.

## 1. Chọn đợt

In [ ]:
DOT = "dot_01"          # đổi thành đợt cần chạy (xem bảng phân công ở trên)
DUNG_SAU_PHUT = 600     # sau 10 giờ thì không gán thêm câu mới, để kịp đóng gói trước giới hạn 12 giờ của Kaggle
MODEL, SERVED = "Qwen/Qwen3-14B-AWQ", "qwen3-14b"
PROMPT, PROMPT_VERSION, PROMPT_SHA = "scripts/llm/prompt_ban5_en_tach", "ban5-en-tach-p3", "e4cc4216433b"

## 2. Kiểm tra GPU, lấy gói, kiểm tra cấu hình đã khóa

In [ ]:
import glob, json, os, shutil, subprocess, sys, time, urllib.request, zipfile
!nvidia-smi -L
import torch
assert torch.cuda.device_count() >= 2, "Cần 2 GPU: Settings > Accelerator > GPU T4 x2"

WORK = "/kaggle/working/nlp"
shutil.rmtree(WORK, ignore_errors=True)
goi = None
for z in glob.glob("/kaggle/input/**/goi_*.zip", recursive=True):
    if f"data/llm/dot/{DOT}.csv" in zipfile.ZipFile(z).namelist():
        goi = z
if goi:
    zipfile.ZipFile(goi).extractall(WORK)
else:  # Kaggle đã tự giải nén gói
    found = [p for p in glob.glob(f"/kaggle/input/**/data/llm/dot/{DOT}.csv", recursive=True)]
    assert found, f"Không thấy {DOT} trong gói ở /kaggle/input: kiểm tra Add Input, và {DOT} có phải đợt của bạn không"
    shutil.copytree(found[0].rsplit("/data/llm/dot/", 1)[0], WORK)
os.chdir(WORK)
sys.path.insert(0, "scripts/llm")
import chung
_, ver, sha = chung.load_prompt_tach(PROMPT)
print("Gói:", goi or "(đã giải nén)", "| prompt", ver, sha)
assert (ver, sha) == (PROMPT_VERSION, PROMPT_SHA), "Prompt trong gói khác bản đã khóa: dừng, báo Triệu"
import pandas as pd
pc = pd.read_csv("data/llm/dot/phan_cong.csv", dtype=str).set_index("dot")
SPLIT = pc.at[DOT, "split"]
CO_TEST = bool((pd.read_csv(f"data/llm/dot/{DOT}.csv", dtype=str).split == "test").any())
print(f"{DOT}: split {SPLIT}{' (có câu tập test)' if CO_TEST else ''}, {pc.at[DOT, 'so_cau']} câu, người chạy {pc.at[DOT, 'nguoi']}, ước {pc.at[DOT, 'uoc_gio']} giờ")

## 3. Lấy kết quả dở của lần chạy trước (nếu có)
Nếu Input có file kết quả của đợt này (Output của phiên bản trước), chép vào để chỉ gán các câu còn thiếu.

In [ ]:
KQ = f"data/llm/ket_qua/vllm_{SERVED}_mac-dinh_{PROMPT_VERSION}_{DOT}"
os.makedirs("data/llm/ket_qua", exist_ok=True)
cu = sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(KQ)}.jsonl", recursive=True), key=os.path.getsize)
if cu:
    shutil.copy(cu[-1], f"{KQ}.jsonl")
    print("Chạy tiếp từ", cu[-1], "-", sum(1 for _ in open(f"{KQ}.jsonl", encoding="utf-8")), "dòng")
else:
    print("Chạy mới từ đầu")

## 4. Cài vLLM 0.34.0 (phiên bản đã khóa)
Nếu báo xung đột torch hay CUDA: Run › Restart & clear cell outputs rồi chạy lại từ đầu.

In [ ]:
!pip install -q vllm==0.34.0 openai 2>&1 | tail -3
!python -c "import vllm, openai; print('vllm', vllm.__version__, '| openai', openai.__version__)"

## 5. Khởi động máy chủ vLLM
Log ghi vào `/kaggle/working/vllm.log`. Ô này chờ đến khi máy chủ trả lời, tối đa 40 phút.

In [ ]:
import signal
cmd = ["vllm", "serve", MODEL, "--quantization", "awq", "--dtype", "half", "--tensor-parallel-size", "2",
       "--disable-custom-all-reduce", "--max-model-len", "16384", "--gpu-memory-utilization", "0.90",
       "--enable-prefix-caching", "--served-model-name", SERVED, "--port", "8000", "--seed", "0"]
print(" ".join(cmd))
server = subprocess.Popen(cmd, stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5).read()
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 2400:
            print(open("/kaggle/working/vllm.log", encoding="utf-8", errors="replace").read()[-3000:])
            raise SystemExit("vLLM không khởi động được: xem log ở trên")
        time.sleep(15)
body = {"model": SERVED, "temperature": 0, "max_tokens": 20, "messages": [{"role": "user", "content": "Trả lời đúng một từ: OK"}],
        "chat_template_kwargs": {"enable_thinking": False}}
req = urllib.request.Request("http://localhost:8000/v1/chat/completions", json.dumps(body).encode(), {"Content-Type": "application/json"})
tra_loi = json.load(urllib.request.urlopen(req, timeout=120))["choices"][0]["message"]["content"]
print(f"Sẵn sàng sau {time.time() - t0:.0f} giây; hỏi thử: {tra_loi!r}")
assert "ok" in tra_loi.lower(), "Câu trả lời thử không đúng: xem /kaggle/working/vllm.log"

## 6. Gán
Chạy thử 32 câu đầu và kiểm tra, rồi gán cả đợt vào cùng file kết quả (các câu đã gán được bỏ qua). Tiến độ in mỗi 500 câu.

In [ ]:
GAN = (f"python scripts/llm/gan_nhan_llm.py --ds data/llm/dot/{DOT}.csv --nha-cung-cap vllm --model {SERVED} "
       f"--nhiet-do 0 --luong 16 --khong-suy-nghi --prompt {PROMPT} --dung-sau-phut {DUNG_SAU_PHUT}"
       + (" --co-tap-test" if CO_TEST else ""))
!{GAN} --gioi-han 32 > /kaggle/working/thu_32.txt
thu = pd.read_json(f"{KQ}.jsonl", lines=True)
pd.set_option("display.max_colwidth", 120)
display(thu.head(5)[["toxicity", "topic", "stance", "emotion_1", "emotion_2", "bang_chung", "da_sua", "loi"]])
assert (thu.loi == "").mean() >= 0.9, "Quá nhiều câu lỗi khi chạy thử: xem cột loi và /kaggle/working/vllm.log"

In [ ]:
!{GAN}

## 7. Tổng kết đợt

In [ ]:
s = json.load(open(f"{KQ}.tong_ket.json", encoding="utf-8"))
kq = pd.read_json(f"{KQ}.jsonl", lines=True).drop_duplicates("comment_uid", keep="last")
n_vao = int(pc.at[DOT, "so_cau"])
xong = int((kq.loi == "").sum())
print(f"{DOT}: {xong:,}/{n_vao:,} câu có nhãn, {int((kq.loi != '').sum())} câu lỗi, còn thiếu {n_vao - len(kq):,} câu; "
      f"phiên này {s['giay'] / 3600:.1f} giờ")
print("Prompt", s["prompt_version"], s["prompt_sha"], "| mô hình", s["model_tra_loi"])
for h in ["toxicity", "topic", "stance", "emotion_1"]:
    print(f"  {h:10s}", kq[kq.loi == ""][h].value_counts(normalize=True).round(3).to_dict())
print("  hỏi lại toxicity:", int(kq.da_sua.str.contains("hỏi lại toxicity", na=False).sum()),
      "| hạ nhãn:", int(kq.da_sua.str.contains("→", na=False).sum()),
      "| hỏi lại vì max_tokens:", int(kq.da_sua.str.contains("max_tokens", na=False).sum()))
XONG = xong == n_vao
print("\n==> XONG: tải ket_qua_%s.zip gửi Triệu" % DOT if XONG else
      "\n==> CHƯA XONG: tải ket_qua_%s.zip gửi Triệu, rồi chạy lại đợt này với Input là Output của phiên bản này (xem đầu notebook)" % DOT)

## 8. Đóng gói kết quả, dọn dữ liệu
Tải `ket_qua_<đợt>.zip` từ tab Output. File `.jsonl` cạnh đó để chạy tiếp nếu chưa xong. Ô cuối dừng máy chủ và xóa bản dữ liệu đã giải nén.

In [ ]:
shutil.copy("/kaggle/working/vllm.log", f"data/llm/ket_qua/vllm_{DOT}.log")
out = f"/kaggle/working/ket_qua_{DOT}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"data/llm/ket_qua/*{DOT}*"):
        z.write(f, f)
shutil.copy(f"{KQ}.jsonl", "/kaggle/working/")
print(zipfile.ZipFile(out).namelist(), f"{os.path.getsize(out) / 2**20:.1f} MB")

In [ ]:
try:
    os.killpg(server.pid, signal.SIGTERM)
except Exception:
    pass
os.chdir("/kaggle/working")
shutil.rmtree(WORK, ignore_errors=True)
for f in ["/kaggle/working/vllm.log", "/kaggle/working/thu_32.txt"]:
    if os.path.exists(f):
        os.remove(f)
print("Còn lại trong /kaggle/working:", os.listdir("/kaggle/working"))